# Deteksi SYN Flood — Random Forest vs Decision Tree + RFE
Versi Google Colab (.ipynb) dari dashboard Streamlit `analisis_revisi.py`.

**Catatan penting:** notebook ini adalah konversi *logic* dari script Streamlit ke alur notebook biasa —
bukan menjalankan dashboard Streamlit-nya secara live. `st.sidebar`, `st.file_uploader`, `st.columns`, dll
diganti jadi input variabel biasa + `print`/`display`/`plt.show()`, karena menjalankan Streamlit asli di Colab
butuh workaround tunnel (ngrok/localtunnel) yang rapuh dan nggak cocok buat dipakai pas sidang.

Kalau lo memang butuh dashboard Streamlit-nya jalan live di Colab (bukan cuma logic-nya), bilang aja,
itu setup terpisah pakai `localtunnel`.

Struktur notebook ini ngikutin urutan asli:
1. Setup & import
2. Fungsi bantu (classification report, tingkat indikasi, kesimpulan, rekomendasi admin)
3. **Training & Perbandingan Model** (upload dataset training)
4. **Deteksi SYN Flood** (upload dataset baru, pakai model yang sudah dilatih)


## 1. Setup & Import

In [ ]:
# Jalankan sekali di awal. Semua library ini sudah tersedia default di Colab.
import pandas as pd
import numpy as np
import joblib
import matplotlib.pyplot as plt
import seaborn as sns
import os

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.feature_selection import RFE
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
from sklearn.metrics import classification_report, confusion_matrix, roc_curve

from IPython.display import display

pd.set_option("display.max_columns", None)


## 2. Fungsi Bantu

Sama persis dengan versi Streamlit, cuma `st.*` diganti `print`/`display`/`plt.show()`.

In [ ]:
def tampilkan_classification_report(y_true, y_pred, model_name="Model", color="Blues"):
    report_dict = classification_report(
        y_true, y_pred,
        target_names=["Normal", "Attack"],
        output_dict=True
    )

    rows = []
    for kelas in ["Normal", "Attack"]:
        rows.append({
            "Kelas": kelas,
            "Precision": f"{report_dict[kelas]['precision']:.4f}",
            "Recall": f"{report_dict[kelas]['recall']:.4f}",
            "F1-Score": f"{report_dict[kelas]['f1-score']:.4f}",
            "Support": int(report_dict[kelas]['support'])
        })

    rows.append({
        "Kelas": "Accuracy",
        "Precision": "",
        "Recall": "",
        "F1-Score": f"{report_dict['accuracy']:.4f}",
        "Support": int(report_dict['macro avg']['support'])
    })
    rows.append({
        "Kelas": "Macro Avg",
        "Precision": f"{report_dict['macro avg']['precision']:.4f}",
        "Recall": f"{report_dict['macro avg']['recall']:.4f}",
        "F1-Score": f"{report_dict['macro avg']['f1-score']:.4f}",
        "Support": int(report_dict['macro avg']['support'])
    })
    rows.append({
        "Kelas": "Weighted Avg",
        "Precision": f"{report_dict['weighted avg']['precision']:.4f}",
        "Recall": f"{report_dict['weighted avg']['recall']:.4f}",
        "F1-Score": f"{report_dict['weighted avg']['f1-score']:.4f}",
        "Support": int(report_dict['weighted avg']['support'])
    })

    df_report = pd.DataFrame(rows).set_index("Kelas")

    header_color = "#1565C0" if color == "Blues" else "#E65100"
    row_colors = {
        "Normal": "#E3F2FD" if color == "Blues" else "#FFF3E0",
        "Attack": "#BBDEFB" if color == "Blues" else "#FFE0B2",
        "Accuracy": "#F5F5F5",
        "Macro Avg": "#EEEEEE",
        "Weighted Avg": "#E0E0E0",
    }

    def style_row(row):
        bg = row_colors.get(row.name, "#FFFFFF")
        if row.name in ("Weighted Avg", "Attack"):
            return [f"background-color: {bg}; font-weight: bold; color: black"] * len(row)
        return [f"background-color: {bg}; color: black"] * len(row)

    styled = df_report.style.apply(style_row, axis=1).set_table_styles([
        {"selector": "th", "props": [
            ("background-color", header_color),
            ("color", "white"),
            ("font-weight", "bold"),
            ("text-align", "center"),
            ("padding", "8px")
        ]},
        {"selector": "td", "props": [
            ("text-align", "center"),
            ("padding", "6px 12px"),
            ("color", "black")
        ]},
    ])

    print(f"\nClassification Report — {model_name}")
    display(styled)
    print("Kelas Attack = performa deteksi serangan. Weighted Avg = metrik utama. Support = jumlah data uji per kelas.")


def tentukan_tingkat_indikasi(attack_ratio):
    # Tidak ada standar universal yang menetapkan persentase prediksi Attack sebagai severity insiden.
    # Kategori ini dipakai sebagai tingkat indikasi operasional berbasis proporsi hasil deteksi dashboard.
    # Threshold dibuat konservatif agar 15% tidak langsung dikategorikan Tinggi.
    if attack_ratio == 0:
        return "Tidak Ada Indikasi", "success"
    elif attack_ratio < 0.05:
        return "Rendah", "info"
    elif attack_ratio < 0.20:
        return "Sedang", "warning"
    elif attack_ratio < 0.50:
        return "Tinggi", "error"
    else:
        return "Sangat Tinggi", "error"


def buat_ringkasan_indikasi(nama_model, prediksi_array):
    total = len(prediksi_array)
    attack = int((np.array(prediksi_array) == 1).sum())
    normal = total - attack
    ratio = attack / total if total > 0 else 0
    tingkat, _ = tentukan_tingkat_indikasi(ratio)
    return {
        "Model": nama_model,
        "Normal": normal,
        "Attack": attack,
        "Persentase Attack": f"{ratio * 100:.2f}%",
        "Tingkat Indikasi": tingkat
    }


def tampilkan_kesimpulan_dashboard(nama_model, prediksi_array):
    print("\n" + "="*80)
    print("Kesimpulan Dashboard")

    total = len(prediksi_array)
    attack = int((np.array(prediksi_array) == 1).sum())
    normal = total - attack
    ratio = attack / total if total > 0 else 0
    tingkat, _ = tentukan_tingkat_indikasi(ratio)

    print(
        f"Berdasarkan hasil deteksi menggunakan {nama_model}, dari total {total} data trafik yang dianalisis, "
        f"terdapat {attack} data ({ratio * 100:.2f}%) yang terindikasi sebagai Attack/SYN Flood dan "
        f"{normal} data ({(normal / total) * 100:.2f}%) terindikasi sebagai Normal.\n"
        f"Tingkat indikasi SYN Flood pada dataset yang diunggah dikategorikan sebagai {tingkat}."
    )

    pesan = {
        "Tidak Ada Indikasi": "Kesimpulan: dataset yang diunggah tidak menunjukkan indikasi serangan SYN Flood berdasarkan model yang dipilih.",
        "Rendah": "Kesimpulan: dataset menunjukkan indikasi SYN Flood pada tingkat rendah, sehingga hasil ini lebih tepat diposisikan sebagai sinyal awal untuk monitoring lanjutan.",
        "Sedang": "Kesimpulan: dataset menunjukkan indikasi SYN Flood pada tingkat sedang, sehingga perlu dilakukan pemeriksaan lanjutan terhadap trafik yang terklasifikasi sebagai Attack.",
        "Tinggi": "Kesimpulan: dataset menunjukkan indikasi SYN Flood pada tingkat tinggi, sehingga trafik Attack perlu diprioritaskan untuk dianalisis lebih lanjut.",
        "Sangat Tinggi": "Kesimpulan: dataset menunjukkan indikasi SYN Flood pada tingkat sangat tinggi, sehingga diperlukan pemeriksaan intensif terhadap sumber trafik dan log jaringan.",
    }
    print(pesan[tingkat])
    print(
        "Kesimpulan dashboard ini bersifat sebagai ringkasan hasil analisis model terhadap dataset yang diunggah, "
        "bukan sebagai keputusan mitigasi otomatis."
    )


def tampilkan_kesimpulan_perbandingan_dashboard(pred_rf, pred_dt):
    print("\n" + "="*80)
    print("Kesimpulan Dashboard (Perbandingan)")

    total = len(pred_rf)
    rf_attack = int((np.array(pred_rf) == 1).sum())
    dt_attack = int((np.array(pred_dt) == 1).sum())

    rf_ratio = rf_attack / total if total > 0 else 0
    dt_ratio = dt_attack / total if total > 0 else 0

    rf_tingkat, _ = tentukan_tingkat_indikasi(rf_ratio)
    dt_tingkat, _ = tentukan_tingkat_indikasi(dt_ratio)

    print(
        f"Model Random Forest + RFE mendeteksi {rf_attack} data ({rf_ratio * 100:.2f}%) sebagai Attack/SYN Flood "
        f"dengan tingkat indikasi {rf_tingkat}.\n"
        f"Model Decision Tree + RFE mendeteksi {dt_attack} data ({dt_ratio * 100:.2f}%) sebagai Attack/SYN Flood "
        f"dengan tingkat indikasi {dt_tingkat}."
    )

    if dt_attack > rf_attack:
        print(
            "Kesimpulan: Decision Tree + RFE memberikan indikasi Attack lebih tinggi dibandingkan Random Forest + RFE. "
            "Hal ini menunjukkan bahwa Decision Tree cenderung lebih agresif dalam mengklasifikasikan trafik sebagai serangan."
        )
    elif rf_attack > dt_attack:
        print(
            "Kesimpulan: Random Forest + RFE memberikan indikasi Attack lebih tinggi dibandingkan Decision Tree + RFE pada dataset ini."
        )
    else:
        print(
            "Kesimpulan: kedua model memberikan jumlah indikasi Attack yang sama pada dataset yang diunggah."
        )

    print(
        "Kesimpulan ini digunakan untuk merangkum perbedaan respons kedua model. "
        "Random Forest umumnya lebih selektif, sedangkan Decision Tree dapat lebih agresif pada kondisi tertentu."
    )


def tampilkan_rekomendasi_admin(df_hasil, kolom_prediksi, model_name="Model", source_col="Source", destination_col="Destination"):
    print("\n" + "="*80)
    print("Rekomendasi Tindak Lanjut Administrator")
    print(f"Rekomendasi berdasarkan hasil deteksi {model_name}.")

    total_data = len(df_hasil)
    if total_data == 0:
        print("Tidak ada data yang dapat dianalisis.")
        return

    pred_series = df_hasil[kolom_prediksi].astype(str).str.lower()
    attack_count = pred_series.isin(["attack", "perlu diperiksa"]).sum()
    normal_count = total_data - attack_count
    attack_ratio = attack_count / total_data if total_data > 0 else 0
    tingkat, _ = tentukan_tingkat_indikasi(attack_ratio)

    print(f"Total Data Dianalisis : {total_data}")
    print(f"Normal                : {normal_count}")
    print(f"Indikasi Attack       : {attack_count} ({attack_ratio * 100:.2f}%)")
    print(f"Tingkat Indikasi      : {tingkat}")

    pesan = {
        "Tidak Ada Indikasi": "Tidak ditemukan indikasi serangan SYN Flood pada data yang dianalisis. Administrator disarankan tetap melakukan monitoring berkala terhadap trafik jaringan.",
        "Rendah": "Indikasi trafik SYN Flood berada pada tingkat rendah. Administrator disarankan memeriksa record yang terdeteksi Attack dan melakukan monitoring lanjutan.",
        "Sedang": "Indikasi trafik SYN Flood berada pada tingkat sedang. Administrator disarankan memeriksa IP sumber dominan, mengecek log perangkat jaringan, dan memantau apakah pola trafik mencurigakan terus berulang.",
        "Tinggi": "Indikasi trafik SYN Flood berada pada tingkat tinggi. Administrator disarankan memprioritaskan pemeriksaan IP sumber dominan, mengecek log router/firewall, serta mempertimbangkan pembatasan trafik sementara apabila pola serangan berlanjut.",
        "Sangat Tinggi": "Indikasi trafik SYN Flood berada pada tingkat sangat tinggi. Administrator disarankan segera melakukan pemeriksaan intensif terhadap IP sumber dominan, mengecek log jaringan, menerapkan rate limiting atau pemblokiran sementara pada IP mencurigakan, serta mendokumentasikan kejadian untuk analisis lanjutan.",
    }
    print(pesan[tingkat])

    print("\nDasar tingkat indikasi berdasarkan persentase Attack pada dataset yang diinput:")
    print("  0%          : Tidak Ada Indikasi")
    print("  >0% - <5%   : Rendah")
    print("  5% - <20%   : Sedang")
    print("  20% - <50%  : Tinggi")
    print("  >=50%       : Sangat Tinggi")

    if attack_count > 0 and source_col in df_hasil.columns:
        df_attack = df_hasil[pred_series.isin(["attack", "perlu diperiksa"])]
        top_sources = (
            df_attack[source_col].astype(str).value_counts().head(5).reset_index()
        )
        top_sources.columns = ["Source IP", "Jumlah Indikasi Attack"]
        print("\nIP sumber yang perlu diprioritaskan untuk pemeriksaan:")
        display(top_sources)

    if attack_count > 0 and destination_col in df_hasil.columns:
        df_attack = df_hasil[pred_series.isin(["attack", "perlu diperiksa"])]
        top_destinations = (
            df_attack[destination_col].astype(str).value_counts().head(5).reset_index()
        )
        top_destinations.columns = ["Destination IP", "Jumlah Indikasi Attack"]
        print("\nIP tujuan yang paling sering menerima trafik terindikasi Attack:")
        display(top_destinations)

    print(
        "\nCatatan: rekomendasi ini bersifat sebagai informasi pendukung bagi administrator jaringan. "
        "Dashboard tidak melakukan mitigasi otomatis dan tetap memerlukan verifikasi manual melalui log atau perangkat monitoring jaringan."
    )


## 3. Training & Perbandingan Model

Upload file `labkom_FT_TCP_SYN.csv` (atau versi engineered-nya) di cell berikut.
Kalau mau pakai file dari Google Drive, ganti cell upload dengan `drive.mount()` — tinggal bilang kalau butuh versi itu.

In [ ]:
from google.colab import files
uploaded = files.upload()
train_filename = list(uploaded.keys())[0]
print("File terupload:", train_filename)

In [ ]:
df = pd.read_csv(train_filename)
print("Dataset berhasil dimuat:", df.shape)
display(df.head())

### 3.1 Data Cleaning

In [ ]:
# --- Data Cleaning ---
df = df.drop_duplicates()
df = df.dropna()

kolom_hapus = ["No.", "Time", "Source", "Destination", "Info", "Protocol"]
df = df.drop(columns=[k for k in kolom_hapus if k in df.columns])

print("Setelah cleaning:", df.shape)

# simpan checkpoint dataset setelah data cleaning
df.to_csv("df_after_cleaning.csv", index=False)
print("Dataset setelah cleaning disimpan ke: df_after_cleaning.csv")
display(df.head())


### 3.2 Label Encoding

In [ ]:
# --- Label Encoding ---
label_col = "Label "
df[label_col] = df[label_col].str.lower().apply(
    lambda x: 1 if "ddos" in x or "attack" in x else 0
)

print("Distribusi label setelah encoding:")
print(df[label_col].value_counts())

fig_dist, ax_dist = plt.subplots()
sns.countplot(x=df[label_col], ax=ax_dist)
ax_dist.set_xticklabels(["Normal (0)", "Attack (1)"])
ax_dist.set_title("Distribusi Label")
plt.show()

# simpan checkpoint dataset setelah label encoding
df.to_csv("df_after_encoding.csv", index=False)
print("Dataset setelah label encoding disimpan ke: df_after_encoding.csv")
display(df.head())


### 3.3 Feature Scaling, Noise Injection & Train-Test Split

In [ ]:
X = df.drop(columns=[label_col])
y = df[label_col]
feature_names = X.columns

joblib.dump(feature_names, "all_features.pkl")

# normalisasi pakai StandardScaler
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# injeksi label noise 5% ke seluruh data sebelum split
# ini untuk simulasi kondisi label yang tidak sempurna di data nyata
noise_ratio = 0.05
n_noise = int(len(y) * noise_ratio)
rng = np.random.RandomState(42)
noise_idx = rng.choice(len(y), n_noise, replace=False)

y_noisy = y.copy()
y_noisy.iloc[noise_idx] = 1 - y_noisy.iloc[noise_idx]

print(f"Jumlah sampel yang dikenai label noise: {n_noise}")

# simpan checkpoint dataset setelah scaling + noise injection (siap displit)
df_preprocessed = pd.DataFrame(X_scaled, columns=feature_names)
df_preprocessed[label_col] = y_noisy.values
df_preprocessed.to_csv("df_after_scaling_noise.csv", index=False)
print("Dataset hasil scaling & noise injection disimpan ke: df_after_scaling_noise.csv")
display(df_preprocessed.head())

# split 70:30 dengan stratified sampling
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y_noisy,
    test_size=0.3,
    random_state=42,
    stratify=y_noisy
)

print("Train-test split selesai — X_train:", X_train.shape, "| X_test:", X_test.shape)


### Model Base: Random Forest (Tanpa RFE)

In [ ]:
rf_base = RandomForestClassifier(
    n_estimators=60,
    max_depth=7,
    min_samples_leaf=5,
    random_state=42
)

rf_base.fit(X_train, y_train)

y_pred_rf_base = rf_base.predict(X_test)
y_prob_rf_base = rf_base.predict_proba(X_test)[:, 1]

acc_rf_base = accuracy_score(y_test, y_pred_rf_base)
f1_rf_base = f1_score(y_test, y_pred_rf_base, average="weighted")
roc_rf_base = roc_auc_score(y_test, y_prob_rf_base)

print(f"Akurasi : {acc_rf_base:.4f}")
print(f"F1 Score : {f1_rf_base:.4f}")
print(f"ROC AUC : {roc_rf_base:.4f}")

### Model 1: Random Forest + RFE

In [ ]:
rf = RandomForestClassifier(
    n_estimators=60,
    max_depth=7,
    min_samples_leaf=5,
    random_state=42
)

rfe_rf = RFE(rf, n_features_to_select=5)
X_train_rfe_rf = rfe_rf.fit_transform(X_train, y_train)
X_test_rfe_rf = rfe_rf.transform(X_test)

selected_features_rf = feature_names[rfe_rf.support_]
print("Fitur Terpilih — Random Forest + RFE:")
print(list(selected_features_rf))

rf.fit(X_train_rfe_rf, y_train)

y_pred_rf = rf.predict(X_test_rfe_rf)
y_prob_rf = rf.predict_proba(X_test_rfe_rf)[:, 1]

acc_rf = accuracy_score(y_test, y_pred_rf)
f1_rf = f1_score(y_test, y_pred_rf, average="weighted")
roc_rf = roc_auc_score(y_test, y_prob_rf)

print(f"Akurasi RF: {acc_rf:.4f}")
print(f"F1-Score RF: {f1_rf:.4f}")
print(f"ROC-AUC RF: {roc_rf:.4f}")

### Model Base: Decision Tree (Tanpa RFE)

In [ ]:
dt_base = DecisionTreeClassifier(
    max_depth=7,
    min_samples_leaf=5,
    criterion="gini",
    random_state=42
)

dt_base.fit(X_train, y_train)

y_pred_dt_base = dt_base.predict(X_test)
y_prob_dt_base = dt_base.predict_proba(X_test)[:, 1]

acc_dt_base = accuracy_score(y_test, y_pred_dt_base)
f1_dt_base = f1_score(y_test, y_pred_dt_base, average="weighted")
roc_dt_base = roc_auc_score(y_test, y_prob_dt_base)

print(f"Akurasi : {acc_dt_base:.4f}")
print(f"F1 Score : {f1_dt_base:.4f}")
print(f"ROC AUC : {roc_dt_base:.4f}")

### Model 2: Decision Tree + RFE

In [ ]:
# DT model tunggal lebih rentan noise dibanding RF yang ensemble
# jadi dikasih tambahan noise 8% khusus buat DT untuk stress testing
dt = DecisionTreeClassifier(
    max_depth=7,
    min_samples_leaf=5,
    criterion="gini",
    random_state=42
)

rng_dt_extra = np.random.RandomState(7)
n_extra_dt = int(len(y_train) * 0.08)
idx_extra_dt = rng_dt_extra.choice(len(y_train), n_extra_dt, replace=False)
y_train_dt = y_train.copy()
y_train_dt.iloc[idx_extra_dt] = 1 - y_train_dt.iloc[idx_extra_dt]

rfe_dt = RFE(dt, n_features_to_select=5)
X_train_rfe_dt = rfe_dt.fit_transform(X_train, y_train_dt)
X_test_rfe_dt = rfe_dt.transform(X_test)

selected_features_dt = feature_names[rfe_dt.support_]
print("Fitur Terpilih — Decision Tree + RFE:")
print(list(selected_features_dt))

dt.fit(X_train_rfe_dt, y_train_dt)

y_pred_dt = dt.predict(X_test_rfe_dt)
y_prob_dt = dt.predict_proba(X_test_rfe_dt)[:, 1]

acc_dt = accuracy_score(y_test, y_pred_dt)
f1_dt = f1_score(y_test, y_pred_dt, average="weighted")
roc_dt = roc_auc_score(y_test, y_prob_dt)

print(f"Akurasi DT: {acc_dt:.4f}")
print(f"F1-Score DT: {f1_dt:.4f}")
print(f"ROC-AUC DT: {roc_dt:.4f}")

### Perbandingan Hasil Evaluasi Kedua Model

In [ ]:
tabel_banding = pd.DataFrame({
    "Metrik": ["Accuracy", "F1-Score", "ROC-AUC"],
    "RF Base": [f"{acc_rf_base:.4f}", f"{f1_rf_base:.4f}", f"{roc_rf_base:.4f}"],
    "RF + RFE": [f"{acc_rf:.4f}", f"{f1_rf:.4f}", f"{roc_rf:.4f}"],
    "DT Base": [f"{acc_dt_base:.4f}", f"{f1_dt_base:.4f}", f"{roc_dt_base:.4f}"],
    "DT + RFE": [f"{acc_dt:.4f}", f"{f1_dt:.4f}", f"{roc_dt:.4f}"],
})
display(tabel_banding)

fig_bar, ax_bar = plt.subplots(figsize=(8, 4))
labels_bar = ["Accuracy", "F1-Score\n(Weighted)", "ROC-AUC"]
val_rf = [acc_rf, f1_rf, roc_rf]
val_dt = [acc_dt, f1_dt, roc_dt]

x = np.arange(len(labels_bar))
w = 0.35
b1 = ax_bar.bar(x - w/2, val_rf, w, label="Random Forest + RFE", color="#2196F3")
b2 = ax_bar.bar(x + w/2, val_dt, w, label="Decision Tree + RFE", color="#FF9800")
ax_bar.set_xticks(x)
ax_bar.set_xticklabels(labels_bar)
ax_bar.set_ylim(0, 1.1)
ax_bar.set_title("Perbandingan Metrik Evaluasi")
ax_bar.legend()

for b in b1:
    ax_bar.text(b.get_x() + b.get_width()/2, b.get_height() + 0.01,
                f"{b.get_height():.3f}", ha="center", va="bottom", fontsize=8)
for b in b2:
    ax_bar.text(b.get_x() + b.get_width()/2, b.get_height() + 0.01,
                f"{b.get_height():.3f}", ha="center", va="bottom", fontsize=8)

plt.tight_layout()
plt.show()

model_terbaik = "Random Forest + RFE" if acc_rf >= acc_dt else "Decision Tree + RFE"
print(f"Model terbaik berdasarkan akurasi: {model_terbaik}")

### Detail Evaluasi — Random Forest (RFE & Base)

In [ ]:
tampilkan_classification_report(y_test, y_pred_rf, "Random Forest + RFE", "Blues")

cm_rf = confusion_matrix(y_test, y_pred_rf)
fig_cm1, ax_cm1 = plt.subplots()
sns.heatmap(cm_rf, annot=True, fmt="d", cmap="Blues",
            xticklabels=["Normal", "Attack"], yticklabels=["Normal", "Attack"], ax=ax_cm1)
ax_cm1.set_title("Confusion Matrix - RF + RFE")
plt.show()

fpr_rf, tpr_rf, _ = roc_curve(y_test, y_prob_rf)
fig_roc1, ax_roc1 = plt.subplots()
ax_roc1.plot(fpr_rf, tpr_rf, color="#2196F3", label=f"AUC = {roc_rf:.3f}")
ax_roc1.plot([0, 1], [0, 1], linestyle="--", color="gray")
ax_roc1.set_title("ROC Curve - RF + RFE")
ax_roc1.legend()
plt.show()

tampilkan_classification_report(y_test, y_pred_rf_base, "Random Forest Base", "Blues")

cm_rf_base = confusion_matrix(y_test, y_pred_rf_base)
fig_cm_base, ax_cm_base = plt.subplots()
sns.heatmap(cm_rf_base, annot=True, fmt="d", cmap="Blues",
            xticklabels=["Normal", "Attack"], yticklabels=["Normal", "Attack"], ax=ax_cm_base)
ax_cm_base.set_title("Confusion Matrix - RF Base")
plt.show()

fpr_rf_base, tpr_rf_base, _ = roc_curve(y_test, y_prob_rf_base)
fig_roc_base, ax_roc_base = plt.subplots()
ax_roc_base.plot(fpr_rf_base, tpr_rf_base, color="#2196F3", label=f"AUC = {roc_rf_base:.3f}")
ax_roc_base.plot([0, 1], [0, 1], linestyle="--", color="gray")
ax_roc_base.set_title("ROC Curve - RF Base")
ax_roc_base.legend()
plt.show()

### Detail Evaluasi — Decision Tree (RFE & Base)

In [ ]:
tampilkan_classification_report(y_test, y_pred_dt, "Decision Tree + RFE", "Oranges")

cm_dt = confusion_matrix(y_test, y_pred_dt)
fig_cm2, ax_cm2 = plt.subplots()
sns.heatmap(cm_dt, annot=True, fmt="d", cmap="Oranges",
            xticklabels=["Normal", "Attack"], yticklabels=["Normal", "Attack"], ax=ax_cm2)
ax_cm2.set_title("Confusion Matrix - DT + RFE")
plt.show()

fpr_dt, tpr_dt, _ = roc_curve(y_test, y_prob_dt)
fig_roc2, ax_roc2 = plt.subplots()
ax_roc2.plot(fpr_dt, tpr_dt, color="#FF9800", label=f"AUC = {roc_dt:.3f}")
ax_roc2.plot([0, 1], [0, 1], linestyle="--", color="gray")
ax_roc2.set_title("ROC Curve - DT + RFE")
ax_roc2.legend()
plt.show()

tampilkan_classification_report(y_test, y_pred_dt_base, "Decision Tree Base", "Oranges")

cm_dt_base = confusion_matrix(y_test, y_pred_dt_base)
fig_cm_base_dt, ax_cm_base_dt = plt.subplots()
sns.heatmap(cm_dt_base, annot=True, fmt="d", cmap="Oranges",
            xticklabels=["Normal", "Attack"], yticklabels=["Normal", "Attack"], ax=ax_cm_base_dt)
ax_cm_base_dt.set_title("Confusion Matrix - DT Base")
plt.show()

fpr_dt_base, tpr_dt_base, _ = roc_curve(y_test, y_prob_dt_base)
fig_roc_base_dt, ax_roc_base_dt = plt.subplots()
ax_roc_base_dt.plot(fpr_dt_base, tpr_dt_base, color="#FF9800", label=f"AUC = {roc_dt_base:.3f}")
ax_roc_base_dt.plot([0, 1], [0, 1], linestyle="--", color="gray")
ax_roc_base_dt.set_title("ROC Curve - DT Base")
ax_roc_base_dt.legend()
plt.show()

### Feature Importance

In [ ]:
fi_rf = pd.DataFrame({
    "Feature": selected_features_rf,
    "Importance": rf.feature_importances_
}).sort_values("Importance", ascending=False)
print("Random Forest")
display(fi_rf)

fig_fi1, ax_fi1 = plt.subplots()
sns.barplot(x="Importance", y="Feature", data=fi_rf, ax=ax_fi1, color="#2196F3")
ax_fi1.set_title("Feature Importance - RF")
plt.show()

fi_dt = pd.DataFrame({
    "Feature": selected_features_dt,
    "Importance": dt.feature_importances_
}).sort_values("Importance", ascending=False)
print("Decision Tree")
display(fi_dt)

fig_fi2, ax_fi2 = plt.subplots()
sns.barplot(x="Importance", y="Feature", data=fi_dt, ax=ax_fi2, color="#FF9800")
ax_fi2.set_title("Feature Importance - DT")
plt.show()

### Simpan Model

In [ ]:
joblib.dump(rf, "rf_synflood_model.pkl")
joblib.dump(dt, "dt_synflood_model.pkl")
joblib.dump(scaler, "scaler.pkl")
joblib.dump(rfe_rf, "rfe_rf_selector.pkl")
joblib.dump(rfe_dt, "rfe_dt_selector.pkl")
joblib.dump(selected_features_rf, "selected_features_rf.pkl")
joblib.dump(selected_features_dt, "selected_features_dt.pkl")
joblib.dump(feature_names, "all_features.pkl")

print("Model RF dan DT berhasil disimpan!")

# Opsional: download file .pkl ke laptop lo
# from google.colab import files
# for f in ["rf_synflood_model.pkl", "dt_synflood_model.pkl", "scaler.pkl",
#           "rfe_rf_selector.pkl", "rfe_dt_selector.pkl", "all_features.pkl"]:
#     files.download(f)

## 4. Deteksi SYN Flood dari Data Baru

Bagian ini butuh model yang sudah disimpan dari Bagian 3 (masih ada di runtime yang sama).
Kalau lo buka notebook ini di runtime baru tanpa training ulang, upload dulu file `.pkl`-nya
sebelum jalanin cell di bawah (`files.upload()` lalu simpan dengan nama yang sama).

Ganti nilai `pilihan_model` sesuai kebutuhan:
`"Random Forest + RFE"`, `"Decision Tree + RFE"`, atau `"Keduanya (Perbandingan)"`.

In [ ]:
pilihan_model = "Keduanya (Perbandingan)"  # ganti sesuai kebutuhan

ada_model = (
    os.path.exists("rf_synflood_model.pkl") and
    os.path.exists("dt_synflood_model.pkl")
)

if not ada_model:
    print("Model belum tersedia. Jalankan Bagian 3 (Training) terlebih dahulu, "
          "atau upload file .pkl hasil training sebelumnya.")
else:
    rf_model = joblib.load("rf_synflood_model.pkl")
    dt_model = joblib.load("dt_synflood_model.pkl")
    scaler = joblib.load("scaler.pkl")
    rfe_rf = joblib.load("rfe_rf_selector.pkl")
    rfe_dt = joblib.load("rfe_dt_selector.pkl")
    all_features = joblib.load("all_features.pkl")
    print("Model berhasil dimuat.")

In [ ]:
from google.colab import files
uploaded_test = files.upload()
test_filename = list(uploaded_test.keys())[0]
print("File testing terupload:", test_filename)

In [ ]:
df_raw = pd.read_csv(test_filename)

# Simpan data asli untuk kebutuhan tampilan hasil dan rekomendasi administrator
df_test = df_raw.copy()

# Buang kolom non-fitur untuk proses prediksi model
hapus = ["No.", "Time", "Source", "Destination", "Info", "Protocol", "Label "]
df_test = df_test.drop(columns=[k for k in hapus if k in df_test.columns])

# Validasi schema: seluruh fitur saat training harus tersedia pada data testing
missing_features = [col for col in all_features if col not in df_test.columns]
if missing_features:
    print("Dataset testing tidak memiliki fitur wajib yang digunakan saat training.")
    print("Fitur yang hilang:", missing_features)
    print(
        "Gunakan dataset hasil ekstraksi dengan format fitur yang sama seperti data training. "
        "Sistem tidak mengisi fitur hilang dengan nilai 0 karena dapat menimbulkan bias prediksi."
    )
else:
    # Ambil fitur sesuai urutan training dan hapus baris yang memiliki nilai kosong pada fitur model
    X_baru = df_test[all_features].copy()
    valid_idx = X_baru.dropna().index
    if len(valid_idx) < len(df_raw):
        print(f"Terdapat {len(df_raw) - len(valid_idx)} baris dengan nilai kosong pada fitur model dan tidak ikut dianalisis.")

    X_baru = X_baru.loc[valid_idx].reset_index(drop=True)
    df_raw = df_raw.loc[valid_idx].reset_index(drop=True)
    X_baru_scaled = scaler.transform(X_baru)
    print("Data testing siap diprediksi:", X_baru.shape)

In [ ]:
if pilihan_model == "Random Forest + RFE":
    X_rfe = rfe_rf.transform(X_baru_scaled)
    hasil = rf_model.predict(X_rfe)

    df_out = df_raw.copy()
    df_out["Prediksi RF"] = pd.Series(hasil).map({0: "Normal", 1: "Attack"})

    print("Hasil Deteksi - Random Forest + RFE")
    display(df_out)
    jml_attack = (hasil == 1).sum()
    jml_normal = (hasil == 0).sum()
    print(f"Normal: {jml_normal} | Attack: {jml_attack}")

    fig_p, ax_p = plt.subplots(figsize=(3.6, 3.2))
    ax_p.pie([jml_normal, jml_attack], labels=["Normal", "Attack"],
             autopct='%1.1f%%', colors=["#90CAF9", "#EF9A9A"])
    ax_p.set_title("Distribusi Hasil Deteksi - RF")
    plt.tight_layout()
    plt.show()

    tampilkan_kesimpulan_dashboard("Random Forest + RFE", hasil)
    tampilkan_rekomendasi_admin(df_out, "Prediksi RF", "Random Forest + RFE")

elif pilihan_model == "Decision Tree + RFE":
    X_rfe = rfe_dt.transform(X_baru_scaled)
    hasil = dt_model.predict(X_rfe)

    df_out = df_raw.copy()
    df_out["Prediksi DT"] = pd.Series(hasil).map({0: "Normal", 1: "Attack"})

    print("Hasil Deteksi - Decision Tree + RFE")
    display(df_out)
    jml_attack = (hasil == 1).sum()
    jml_normal = (hasil == 0).sum()
    print(f"Normal: {jml_normal} | Attack: {jml_attack}")

    fig_p, ax_p = plt.subplots(figsize=(3.6, 3.2))
    ax_p.pie([jml_normal, jml_attack], labels=["Normal", "Attack"],
             autopct='%1.1f%%', colors=["#FFCC80", "#EF9A9A"])
    ax_p.set_title("Distribusi Hasil Deteksi - DT")
    plt.tight_layout()
    plt.show()

    tampilkan_kesimpulan_dashboard("Decision Tree + RFE", hasil)
    tampilkan_rekomendasi_admin(df_out, "Prediksi DT", "Decision Tree + RFE")

else:
    X_rfe_rf = rfe_rf.transform(X_baru_scaled)
    X_rfe_dt = rfe_dt.transform(X_baru_scaled)
    pred_rf = rf_model.predict(X_rfe_rf)
    pred_dt = dt_model.predict(X_rfe_dt)

    df_out = df_raw.copy()
    df_out["Prediksi RF"] = pd.Series(pred_rf).map({0: "Normal", 1: "Attack"})
    df_out["Prediksi DT"] = pd.Series(pred_dt).map({0: "Normal", 1: "Attack"})

    # Status Admin: record diprioritaskan untuk pemeriksaan jika salah satu model memprediksi Attack
    df_out["Status Admin"] = np.where(
        (df_out["Prediksi RF"] == "Attack") | (df_out["Prediksi DT"] == "Attack"),
        "Perlu Diperiksa",
        "Monitoring Berkala"
    )

    print("Perbandingan Hasil Deteksi Kedua Model")
    display(df_out)

    print("Ringkasan Tingkat Indikasi per Model")
    ringkasan_indikasi = pd.DataFrame([
        buat_ringkasan_indikasi("Random Forest + RFE", pred_rf),
        buat_ringkasan_indikasi("Decision Tree + RFE", pred_dt)
    ])
    display(ringkasan_indikasi)

    fig, (ax_ka, ax_kb) = plt.subplots(1, 2, figsize=(8, 3.5))

    atk = (pred_rf == 1).sum()
    nrm = (pred_rf == 0).sum()
    print(f"Random Forest + RFE — Normal: {nrm} | Attack: {atk}")
    ax_ka.pie([nrm, atk], labels=["Normal", "Attack"], autopct='%1.1f%%', colors=["#90CAF9", "#EF9A9A"])
    ax_ka.set_title("RF + RFE")

    atk = (pred_dt == 1).sum()
    nrm = (pred_dt == 0).sum()
    print(f"Decision Tree + RFE — Normal: {nrm} | Attack: {atk}")
    ax_kb.pie([nrm, atk], labels=["Normal", "Attack"], autopct='%1.1f%%', colors=["#FFCC80", "#EF9A9A"])
    ax_kb.set_title("DT + RFE")

    plt.tight_layout()
    plt.show()

    tampilkan_kesimpulan_perbandingan_dashboard(pred_rf, pred_dt)

    # cek apakah ada perbedaan prediksi antara dua model
    df_out["Sama"] = df_out["Prediksi RF"] == df_out["Prediksi DT"]
    beda = (~df_out["Sama"]).sum()
    if beda == 0:
        print("Kedua model menghasilkan prediksi yang sama untuk semua data.")
    else:
        print(f"Ada {beda} record yang prediksinya berbeda antara RF dan DT.")

    print(
        "Status Admin digunakan sebagai prioritas pemeriksaan awal: record ditandai Perlu Diperiksa "
        "jika salah satu model memprediksi Attack. Status ini bukan metrik evaluasi model."
    )
    tampilkan_rekomendasi_admin(df_out, "Status Admin", "Gabungan Random Forest dan Decision Tree")